# B5 v5 FINAL — Confirmatory two-basket IV application (Paper 2, JBES)

This final reproducibility notebook contains the frozen post-audit specification used for the reported equal-weight and PC1 robustness runs:

1. **Fast certified AR projection.** The rowwise analytic HAC Anderson--Rubin sets and the reduced-form coefficient ellipsoids are unchanged. The repeated conic Collatz--Wielandt support solves are replaced by closed-form ellipsoidal support functions, with one feasibility SOCP per design. The resulting interval remains a certified outer interval; no grid truncation or coefficient cap is used.
2. **Corrected temporal placebo.** The old lead implementation shifted both baskets to the future and then tested their mutual first stage, which mechanically remains strong under common-factor persistence. The corrected placebo tests whether the one-minute-ahead basket proxy adds first-stage information for the primary lag-1 exposure **after conditioning on** the contemporaneous basket and the primary lag-1 instrument. It is a falsification diagnostic and does not alter the primary lag 1/2/5/10 specifications.

Equal-weight baskets are primary; `pc1_train` is a robustness run. Three pre-specified balanced partitions are retained. HAC bandwidth is 12 bins. No scalar F threshold authorizes plug-in inference; the AR interval is reported for every non-placebo design.

Keep `QUICK_MODE=True` for a complete end-to-end validation with actual AR intervals. Change to `False` only after the quick audit passes.


In [ ]:
QUICK_MODE = True
APP_QUICK_MODE = QUICK_MODE
import os
if os.path.exists('pipeline_v4.py'):
    exec(open('pipeline_v4.py').read())
else:
    exec(open('harness_loader_v4.py').read())
# pipeline_v4.py has its own historical QUICK_MODE=False; restore the application switch.
QUICK_MODE = APP_QUICK_MODE
SYMBOLS = SYMBOLS_QUICK if QUICK_MODE else SYMBOLS_FULL
WINDOWS = [
    next(w for w in WINDOWS_FULL if w["name"] == "FTX_collapse"),
    next(w for w in WINDOWS_FULL if w["name"] == "Control_2022_Aug_A"),
] if QUICK_MODE else WINDOWS_FULL
# Optional absolute raw-data override. Set CRYPTO_RAW_BINANCE in the shell; no downloads are attempted.
if os.environ.get("CRYPTO_RAW_BINANCE"):
    RAW = Path(os.environ["CRYPTO_RAW_BINANCE"]).expanduser()
    RUN_DOWNLOAD = False
exec(open('cascade_cs.py').read())
import pandas as pd, numpy as np
print('pipeline + inference module loaded')
print('application QUICK_MODE:', QUICK_MODE, '| windows:', [w['name'] for w in WINDOWS])
print('RAW:', RAW, '| RUN_DOWNLOAD:', RUN_DOWNLOAD)

# Confirmatory run controls
RESUME = False   # set True only to resume an interrupted run produced by this same v4 notebook
AR_NX = 16 if QUICK_MODE else 64  # only tightens the certified outer projection; validity does not depend on nx


In [ ]:

PANEL_SYMBOLS = list(SYMBOLS)
EXTERNAL = ["XLMUSDT","EOSUSDT","ALGOUSDT","AAVEUSDT","THETAUSDT","VETUSDT","ENJUSDT","MANAUSDT"]
SPLITS = {  # three pre-specified balanced disjoint partitions (match on liquidity/volatility before freezing)
  "S1": (["XLMUSDT","EOSUSDT","ALGOUSDT","AAVEUSDT"], ["THETAUSDT","VETUSDT","ENJUSDT","MANAUSDT"]),
  "S2": (["XLMUSDT","THETAUSDT","ALGOUSDT","ENJUSDT"], ["EOSUSDT","VETUSDT","AAVEUSDT","MANAUSDT"]),
  "S3": (["XLMUSDT","VETUSDT","AAVEUSDT","ENJUSDT"],  ["EOSUSDT","THETAUSDT","ALGOUSDT","MANAUSDT"]),
}
SPLIT_3WAY = (["XLMUSDT","EOSUSDT","ALGOUSDT"], ["AAVEUSDT","THETAUSDT","VETUSDT"], ["ENJUSDT","MANAUSDT"])
for a,b in SPLITS.values(): assert not (set(a)|set(b)) & set(PANEL_SYMBOLS) and not set(a)&set(b)
DESIGN = dict(event_type="volume_burst", bin="1min", lags=60, half_life=10)
Q_GRID = [0.95, 0.97]
PROXY_METHOD = "equal_weight"      # primary; "pc1_train" = PC1 with loadings from the training bins (robustness)
LAGS = [1, 2, 5, 10]; PLACEBO_LEAD = 1
HAC_LAGS = 12; ALPHA = 0.05
TIME_KNOTS = 8                     # flexible intraday time baseline (cubic B-spline over the window), partialled out everywhere
TRAIN_FRAC = 0.3                   # first 30% of the window bins used only to estimate PC1 loadings (pc1_train)
print("splits:", list(SPLITS), "| proxy:", PROXY_METHOD, "| lags:", LAGS, "| placebo lead:", PLACEBO_LEAD)

RUN_ID = f"{PROXY_METHOD}_{'quick' if QUICK_MODE else 'full'}"
print("run id:", RUN_ID, "| AR trial vectors:", AR_NX, "| resume:", RESUME)


## Proxy construction

In [ ]:

def basket_scores(symbols, window):
    if isinstance(symbols, str):
        symbols = [symbols]
    panel_ext = load_window_panel(symbols, window, bin_size=DESIGN["bin"])
    present = [s for s in symbols if s in set(panel_ext["symbol"])]
    if len(present) < 1: raise RuntimeError(f"basket {symbols}: {len(present)} symbols with data")
    act = continuous_activity_score(panel_to_wide_features(panel_ext, present))     # T x |basket| standardized activity
    return act

def basket_proxy(act, method):
    """Basket proxy used in the frozen application.

    equal_weight:
        Cross-sectional mean of standardized basket activity.

    pc1_train:
        First principal component with loadings estimated on the first
        TRAIN_FRAC share of the window.  Because the per-asset activity
        series are aligned on the union of minute bins, missing standardized
        bins can occur.  For PC1 only, those bins are filled with the
        corresponding training-period mean in standardized coordinates
        before SVD/projection.  A deterministic eigh fallback is used only
        if SVD fails numerically.  This is the implementation used for the
        frozen PC1 robustness outputs.
    """
    X = act.astype(float).replace([np.inf, -np.inf], np.nan)
    mu = X.mean(axis=0)
    sd = X.std(axis=0).where(lambda s: s > 1e-12, np.nan)
    Z = (X - mu) / sd

    if method == "equal_weight":
        return pd.Series(Z.mean(axis=1, skipna=True).fillna(0.0).to_numpy(), index=act.index)

    if method != "pc1_train":
        raise ValueError(f"Unknown proxy method: {method}")

    n_tr = int(TRAIN_FRAC * len(Z))
    if n_tr < 2:
        raise RuntimeError(f"Too few training observations for PC1: n_tr={n_tr}")

    Ztr_df = Z.iloc[:n_tr].copy()
    finite_counts = Ztr_df.notna().sum(axis=0)
    bad_cols = finite_counts[finite_counts < 2].index.tolist()
    if bad_cols:
        raise RuntimeError(f"PC1 basket has insufficient finite training observations for columns: {bad_cols}")

    train_fill = Ztr_df.mean(axis=0)
    if not np.isfinite(train_fill.to_numpy(dtype=float)).all():
        raise RuntimeError("Non-finite PC1 training fill values")

    Ztr = Ztr_df.fillna(train_fill).to_numpy(dtype=float)
    Zall = Z.fillna(train_fill).to_numpy(dtype=float)
    if not np.isfinite(Ztr).all() or not np.isfinite(Zall).all():
        raise RuntimeError("Non-finite values remain in PC1 matrices")

    center = Ztr.mean(axis=0)
    Ztr_centered = Ztr - center
    try:
        _, _, Vt = np.linalg.svd(Ztr_centered, full_matrices=False)
        w = Vt[0]
    except np.linalg.LinAlgError:
        C = Ztr_centered.T @ Ztr_centered
        eigvals, eigvecs = np.linalg.eigh(C)
        w = eigvecs[:, np.argmax(eigvals)]

    if np.sum(w) < 0:
        w = -w
    score = Zall @ w
    if not np.isfinite(score).all():
        raise RuntimeError("Non-finite PC1 scores after projection")
    return pd.Series(score, index=act.index)

def shift_aligned(m_series, index, lag):
    m = m_series.reindex(index).ffill().fillna(0.0).to_numpy()
    if lag > 0:  return np.concatenate([np.zeros(lag), m[:-lag]])   # lagged proxy
    if lag < 0:  return np.concatenate([m[-lag:], np.zeros(-lag)])  # LEAD (placebo)
    return m


def time_basis(T, n_knots=8):
    """centered cubic B-spline basis over the window (intraday/secular time baseline), T x (n_knots+2)"""
    from scipy.interpolate import BSpline
    t = np.linspace(0, 1, T); knots = np.concatenate([[0]*4, np.linspace(0, 1, n_knots + 2)[1:-1], [1]*4])
    B = np.column_stack([BSpline(knots, np.eye(n_knots + 4)[j], 3)(t) for j in range(n_knots + 4)])
    B = B[:, 1:-1]; return B - B.mean(0)


## Run

In [ ]:
OUTB = OUT / "tables"
OUTB.mkdir(parents=True, exist_ok=True)

RESULTS_FINAL = OUTB / f"b5v4_instrument_results__{RUN_ID}.csv"
OVERID_FINAL = OUTB / f"b5v4_overid_by_asset__{RUN_ID}.csv"
RESULTS_PARTIAL = OUTB / f"b5v4_instrument_results__{RUN_ID}_PARTIAL.csv"
OVERID_PARTIAL = OUTB / f"b5v4_overid_by_asset__{RUN_ID}_PARTIAL.csv"

# Safe resume: only use non-empty partial files produced by this same v4 notebook/run-id.
def _read_partial(path):
    if (not path.exists()) or path.stat().st_size <= 1:
        return []
    try:
        return pd.read_csv(path).to_dict("records")
    except pd.errors.EmptyDataError:
        return []

rows = _read_partial(RESULTS_PARTIAL) if RESUME else []
rows_ar = _read_partial(OVERID_PARTIAL) if RESUME else []

done = {
    (str(r["window"]), float(r["q"]), str(r["split"]), int(r["lag"]))
    for r in rows
}
done_overid = {
    (str(r["window"]), float(r["q"]), str(r["asset"]))
    for r in rows_ar
}

def checkpoint():
    if rows:
        pd.DataFrame(rows).to_csv(RESULTS_PARTIAL, index=False)
    if rows_ar:
        pd.DataFrame(rows_ar).to_csv(OVERID_PARTIAL, index=False)

print("OUTPUT:", OUTB)
print("RUN_ID:", RUN_ID)
print("existing designs on resume:", len(done))

for w in WINDOWS:
    print("\n" + "=" * 78)
    print("WINDOW", w["name"], "|", w["kind"])
    print("=" * 78, flush=True)

    panel = load_window_panel(PANEL_SYMBOLS, w, bin_size=DESIGN["bin"])
    if panel.empty:
        print("skip empty panel", w["name"])
        continue

    feats = panel_to_wide_features(panel, PANEL_SYMBOLS)

    try:
        acts = {s: basket_scores(s, w) for s in EXTERNAL}
    except Exception as e:
        print("skip externals", w["name"], e)
        continue

    def proxy_for(symbols):
        act = pd.concat([acts[s] for s in symbols], axis=1)
        return basket_proxy(act, PROXY_METHOD)

    for q in Q_GRID:
        Y_df, _ = build_events_from_features(
            feats,
            event_type=DESIGN["event_type"],
            q=q,
        )

        keep = [
            s for s in Y_df.columns
            if Y_df[s].sum() >= MIN_EVENT_COUNT_PER_ASSET
        ]
        if len(keep) < 3:
            print("skip q for eligibility", w["name"], q, "K=", len(keep))
            continue

        Y_df = Y_df[keep]
        dummy = pd.DataFrame(0.0, index=Y_df.index, columns=keep)
        Y, H, _, idx = effective_design(
            Y_df,
            dummy,
            DESIGN["lags"],
            DESIGN["half_life"],
        )
        T, Kk = Y.shape

        Zt = time_basis(T, TIME_KNOTS)
        H = np.column_stack([H, Zt])

        Bn, _, _, _ = fit_rows(Y, H, None)
        rho_naive = spectral_radius(np.maximum(Bn, 0))
        cn = connectedness(np.maximum(Bn, 0))

        print(
            f"q={q:.2f} K={Kk} T={T} rho_naive={rho_naive:.4f}",
            flush=True,
        )

        for split, (bA, bB) in SPLITS.items():
            mA_s = proxy_for(bA)
            mB_s = proxy_for(bB)

            # -------------------------------------------------------------
            # PRIMARY / ROBUSTNESS NETWORK DESIGNS: lags 1,2,5,10
            # -------------------------------------------------------------
            for lag in LAGS:
                key = (w["name"], float(q), split, int(lag))
                if key in done:
                    print("resume skip", key)
                    continue

                import time
                tic = time.perf_counter()

                mA = shift_aligned(mA_s, Y_df.index, lag)[-T:]
                mB = shift_aligned(mB_s, Y_df.index, lag)[-T:]

                kA, _, _ = separation_share(H, mA)
                kB, _, _ = separation_share(H, mB)
                F_conv = effective_F(mA, mB, H, 0)
                F_eff = effective_F(mA, mB, H, HAC_LAGS)

                Bo, _, _, _ = fit_rows(Y, H, mA)
                rho_ols = spectral_radius(np.maximum(Bo, 0))

                B2, _, _ = two_stage_ls(Y, H, mA, mB)
                B2p = np.maximum(B2, 0)
                rho_2sls = spectral_radius(B2p)
                c2s = connectedness(B2p)

                rank_same = (
                    (not np.isnan(cn["total"]))
                    and (not np.isnan(c2s["total"]))
                    and np.array_equal(
                        np.argsort(cn["systemic"]),
                        np.argsort(c2s["systemic"]),
                    )
                )

                kinds = [
                    anderson_rubin_analytic(
                        Y[:, z], H, mA, mB,
                        ALPHA / 2 / Kk,
                        hac_lags=HAC_LAGS,
                    )[0]
                    for z in range(Kk)
                ]

                arp = ar_projected_interval(
                    Y, H, mA, mB,
                    alpha=ALPHA,
                    nx=AR_NX,
                    seed=RANDOM_SEED,
                    hac_lags=HAC_LAGS,
                )

                elapsed = time.perf_counter() - tic

                rows.append(dict(
                    window=w["name"],
                    kind=w["kind"],
                    q=q,
                    split=split,
                    lag=lag,
                    placebo=False,
                    placebo_definition="",
                    K=Kk,
                    T=T,
                    kappa_A=kA,
                    kappa_B=kB,
                    F_conv=F_conv,
                    F_eff_MOP=F_eff,
                    rho_naive=rho_naive,
                    rho_ols_proxy=rho_ols,
                    rho_2sls=rho_2sls,
                    ar_lo=arp["lo"],
                    ar_hi=arp["hi"],
                    ar_empty=arp["empty"],
                    ar_frac_bounded=float(np.mean(arp["ar_bounded"])),
                    hac_ar_frac_bounded=float(np.mean([k == "interval" for k in kinds])),
                    ar_projection=arp.get("projection", "unknown"),
                    conn_naive=cn["total"],
                    conn_2sls=c2s["total"],
                    systemic_rank_preserved=rank_same,
                    seconds=elapsed,
                ))
                done.add(key)
                checkpoint()

                hi_txt = "inf" if not np.isfinite(arp["hi"]) else f"{arp['hi']:.3f}"
                print(
                    f"{w['name']:16s} q={q:.2f} {split} lag={lag:2d} "
                    f"F={F_conv:7.1f} Feff={F_eff:7.1f} | "
                    f"rho {rho_naive:.3f}->{rho_2sls:.3f} | "
                    f"AR [{arp['lo']:.3f},{hi_txt}] "
                    f"bounded={np.mean(arp['ar_bounded']):.0%} "
                    f"empty={arp['empty']} | {elapsed:.2f}s",
                    flush=True,
                )

            # -------------------------------------------------------------
            # CORRECTED TEMPORAL PLACEBO
            # -------------------------------------------------------------
            # Primary exposure: mA(t-1).
            # Primary instrument: mB(t-1).
            # Placebo instrument: mB(t+1), tested only for INCREMENTAL first-stage
            # relevance after conditioning on H, mB(t), and mB(t-1).
            # This isolates future information rather than the persistent common factor
            # shared by two jointly shifted future baskets.
            pkey = (w["name"], float(q), split, -PLACEBO_LEAD)
            if pkey not in done:
                import time
                tic = time.perf_counter()

                mA_primary = shift_aligned(mA_s, Y_df.index, 1)[-T:]
                mB_primary = shift_aligned(mB_s, Y_df.index, 1)[-T:]
                mB_now = shift_aligned(mB_s, Y_df.index, 0)[-T:]
                mB_lead = shift_aligned(mB_s, Y_df.index, -PLACEBO_LEAD)[-T:]

                H_placebo = np.column_stack([H, mB_primary, mB_now])
                kA_p, _, _ = separation_share(H, mA_primary)
                kLead, _, _ = separation_share(H_placebo, mB_lead)

                Fp_conv = effective_F(mA_primary, mB_lead, H_placebo, 0)
                Fp_eff = effective_F(mA_primary, mB_lead, H_placebo, HAC_LAGS)
                elapsed = time.perf_counter() - tic

                rows.append(dict(
                    window=w["name"],
                    kind=w["kind"],
                    q=q,
                    split=split,
                    lag=-PLACEBO_LEAD,
                    placebo=True,
                    placebo_definition="lead_innovation_F:mA_t-1~mB_t+1|H,mB_t,mB_t-1",
                    K=Kk,
                    T=T,
                    kappa_A=kA_p,
                    kappa_B=kLead,
                    F_conv=Fp_conv,
                    F_eff_MOP=Fp_eff,
                    rho_naive=np.nan,
                    rho_ols_proxy=np.nan,
                    rho_2sls=np.nan,
                    ar_lo=np.nan,
                    ar_hi=np.nan,
                    ar_empty=False,
                    ar_frac_bounded=np.nan,
                    hac_ar_frac_bounded=np.nan,
                    ar_projection="not_applicable_placebo",
                    conn_naive=np.nan,
                    conn_2sls=np.nan,
                    systemic_rank_preserved=False,
                    seconds=elapsed,
                ))
                done.add(pkey)
                checkpoint()

                print(
                    f"PLACEBO {w['name']:8s} q={q:.2f} {split}: "
                    f"incremental lead F={Fp_conv:.2f} "
                    f"Feff={Fp_eff:.2f} | {elapsed:.2f}s",
                    flush=True,
                )

        # -------------------------------------------------------------
        # Three-way overidentification, primary lag 1
        # -------------------------------------------------------------
        m3 = [
            shift_aligned(proxy_for(b), Y_df.index, 1)[-T:]
            for b in SPLIT_3WAY
        ]

        for z in range(Kk):
            okey = (w["name"], float(q), str(keep[z]))
            if okey in done_overid:
                continue

            J, pval, dfj = hansen_j_hac(
                Y[:, z], H, m3[0], [m3[1], m3[2]],
                hac_lags=HAC_LAGS,
            )
            rows_ar.append(dict(
                window=w["name"],
                q=q,
                asset=keep[z],
                hansen_J=J,
                p=pval,
                df=dfj,
            ))
            done_overid.add(okey)

        checkpoint()

# -------------------------------------------------------------------------
# Finalize
# -------------------------------------------------------------------------
res = pd.DataFrame(rows).sort_values(["window", "q", "split", "lag"]).reset_index(drop=True)
ov = pd.DataFrame(rows_ar).sort_values(["window", "q", "asset"]).reset_index(drop=True)

res.to_csv(RESULTS_FINAL, index=False)
ov.to_csv(OVERID_FINAL, index=False)

print("\n" + "=" * 78)
print("B5 V4 COMPLETE")
print("=" * 78)
print("RUN_ID:", RUN_ID)
print("results rows:", len(res))
print("non-placebo rows:", int((~res.placebo).sum()))
print("corrected placebo rows:", int(res.placebo.sum()))
print("over-ID rows:", len(ov))
print("results:", RESULTS_FINAL)
print("over-ID:", OVERID_FINAL)
print("partial checkpoints retained:", RESULTS_PARTIAL, OVERID_PARTIAL)

res.round(3)


## Reading the output (pre-registered)
* `F_eff_MOP` is the strength diagnostic; it does **not** authorize plug-in inference. Report `ar_lo/ar_hi` throughout.
* Placebo rows (`placebo=True`, one-minute lead): a lead proxy should carry **no** first stage and produce uninformative AR sets; a strong lead first stage signals a direct basket→panel channel.
* Conclusions are reported only if they hold across `S1,S2,S3` and across lags 1–10.
* `hansen_J` p-values by asset: rejections flag joint-exclusion violations for that asset (diagnostic; unreliable if `F_eff` is small).
* `conn_naive` → `conn_2sls` and `systemic_rank_preserved` implement Proposition 2.1 on the data.